# Deploy zerank-2 on Azure AKS

This notebook shows how to deploy `zerank-2` from Azure Marketplace to an Azure Kubernetes Service (AKS) cluster with an H100 GPU node. You will create the cluster, install the NVIDIA device plugin, deploy the Marketplace extension, and test the local endpoint through `kubectl port-forward`.

## Prerequisites

- An Azure subscription with quota for `Standard_NC40ads_H100_v5`
- Azure CLI (`az`) installed and authenticated with `az login`
- `kubectl` installed
- `jq` installed for JSON inspection
- Permission to create AKS clusters and Azure Kubernetes extensions
- A resource group in the subscription, or permission to create one

## What you will do

- Create a single-node AKS cluster with one H100 GPU
- Install the NVIDIA Kubernetes device plugin
- Deploy `zerank-2` through the Azure Marketplace extension
- Verify the pod, service, GPU allocation, and health endpoint
- Send a sample rerank request to `/invocations`

## 1. Configure deployment values

Set these values before running the shell cells. The Python cell exports them to the notebook process so `%%bash` cells can read them.

In [ ]:
import os

config = {
    "AZURE_RESOURCE_GROUP": "<your-resource-group>",
    "AKS_CLUSTER_NAME": "<your-cluster-name>",
    "AZURE_LOCATION": "WestUS3",
    "AKS_NODE_SIZE": "Standard_NC40ads_H100_v5",
    "AKS_NODE_COUNT": "1",
    "ZERANK_EXTENSION_NAME": "<your-extension-name>",
    "ZERANK_EXTENSION_TYPE": "zeroentropy.extension",
    "ZERANK_PLAN_NAME": "reranker-monthly-plan",
    "ZERANK_PLAN_PRODUCT": "zeroentropy-zerank-2",
    "ZERANK_PLAN_PUBLISHER": "zeroentropy",
    "ZERANK_NAMESPACE": "zerank-2",
    "LOCAL_PORT": "8080",
}

for key, value in config.items():
    os.environ[key] = value

missing = [key for key, value in config.items() if value.startswith("<") and value.endswith(">")]
if missing:
    print("Update these values before running the deployment cells:")
    for key in missing:
        print(f"  - {key}")
else:
    print("Configuration loaded.")

## 2. Create the AKS cluster

Create a single-node AKS cluster backed by `Standard_NC40ads_H100_v5`. If you already have a suitable cluster, skip this cell and run the `az aks get-credentials` cell below.

The cluster location must have quota for both total regional vCPUs and `Standard NCadsH100v5 Family vCPUs`.

In [ ]:
%%bash
set -euo pipefail

az aks create \
  --resource-group "$AZURE_RESOURCE_GROUP" \
  --name "$AKS_CLUSTER_NAME" \
  --location "$AZURE_LOCATION" \
  --node-count "$AKS_NODE_COUNT" \
  --node-vm-size "$AKS_NODE_SIZE" \
  --generate-ssh-keys

Connect `kubectl` to the cluster.

In [ ]:
%%bash
set -euo pipefail

az aks get-credentials \
  --resource-group "$AZURE_RESOURCE_GROUP" \
  --name "$AKS_CLUSTER_NAME" \
  --overwrite-existing

kubectl get nodes

## 3. Install the NVIDIA device plugin

The device plugin advertises GPU resources to Kubernetes. After installation, wait about one minute for the allocatable GPU count to appear on the node.

In [ ]:
%%bash
set -euo pipefail

kubectl apply -f https://raw.githubusercontent.com/NVIDIA/k8s-device-plugin/v0.17.0/deployments/static/nvidia-device-plugin.yml

Verify that Kubernetes sees the GPU. The kubelet may need a minute to refresh node allocatable resources after the plugin registers. This cell retries until it sees `"gpu": "1"`.

In [ ]:
%%bash
set -euo pipefail

for i in {1..12}; do
  gpu_count=$(kubectl get nodes -o json | jq -r '.items[0].status.allocatable["nvidia.com/gpu"] // ""')
  if [ "$gpu_count" = "1" ]; then
    kubectl get nodes -o json | jq '.items[] | {name: .metadata.name, gpu: .status.allocatable["nvidia.com/gpu"]}'
    exit 0
  fi
  echo "Waiting for GPU allocatable resource..."
  sleep 10
done

kubectl get nodes -o json | jq '.items[] | {name: .metadata.name, gpu: .status.allocatable["nvidia.com/gpu"]}'
exit 1

## 4. Deploy zerank-2 from Azure Marketplace

You can deploy through the [zerank-2 Azure Marketplace listing](https://marketplace.microsoft.com/en-us/product/zeroentropy.zeroentropy-zerank-2?tab=Overview) or create the Kubernetes extension from the CLI.

The command below uses the current Azure Marketplace extension type and plan verified for `zerank-2`.

In [ ]:
%%bash
set -euo pipefail

az k8s-extension create \
  --name "$ZERANK_EXTENSION_NAME" \
  --cluster-name "$AKS_CLUSTER_NAME" \
  --resource-group "$AZURE_RESOURCE_GROUP" \
  --cluster-type managedClusters \
  --extension-type "$ZERANK_EXTENSION_TYPE" \
  --scope cluster \
  --release-namespace "$ZERANK_NAMESPACE" \
  --plan-name "$ZERANK_PLAN_NAME" \
  --plan-product "$ZERANK_PLAN_PRODUCT" \
  --plan-publisher "$ZERANK_PLAN_PUBLISHER"

## 5. Verify the deployment

Wait until the pod is ready. The first startup can take several minutes while the image is pulled and the model loads.

In [ ]:
%%bash
set -euo pipefail

kubectl wait \
  --for=condition=ready pod \
  -n "$ZERANK_NAMESPACE" \
  -l app.kubernetes.io/name=zerank-2 \
  --timeout=900s

kubectl get pods -n "$ZERANK_NAMESPACE" -o wide

List services in the namespace. The service name includes the extension instance name.

In [ ]:
%%bash
set -euo pipefail

kubectl get svc -n "$ZERANK_NAMESPACE" -o wide

Check recent logs if the pod is not ready.

In [ ]:
%%bash
set -euo pipefail

kubectl logs -n "$ZERANK_NAMESPACE" -l app.kubernetes.io/name=zerank-2 --tail=100

## 6. Port-forward the service

This starts `kubectl port-forward` from the notebook kernel and discovers the service name automatically.

In [ ]:
import os
import subprocess
import time

namespace = os.environ["ZERANK_NAMESPACE"]
local_port = os.environ["LOCAL_PORT"]

service_name = subprocess.check_output(
    [
        "kubectl",
        "get",
        "svc",
        "-n",
        namespace,
        "-l",
        "app.kubernetes.io/name=zerank-2",
        "-o",
        "jsonpath={.items[0].metadata.name}",
    ],
    text=True,
).strip()

if "zerank2_port_forward" in globals() and zerank2_port_forward.poll() is None:
    zerank2_port_forward.terminate()
    zerank2_port_forward.wait(timeout=10)

zerank2_port_forward = subprocess.Popen(
    [
        "kubectl",
        "port-forward",
        "-n",
        namespace,
        f"svc/{service_name}",
        f"{local_port}:80",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)
if zerank2_port_forward.poll() is not None:
    output = zerank2_port_forward.stdout.read() if zerank2_port_forward.stdout else ""
    raise RuntimeError(f"port-forward exited early:\n{output}")

print(f"Forwarding localhost:{local_port} to service/{service_name}:80")

## 7. Test the rerank endpoint

Send a sample request to `/invocations`. The response returns document indexes sorted by relevance score, highest first.

In [ ]:
%%bash
set -euo pipefail

curl -X POST "http://localhost:${LOCAL_PORT}/invocations" \
  -H "Content-Type: application/json" \
  -d '{
    "query": "What is machine learning?",
    "documents": [
      "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
      "The weather forecast for tomorrow predicts sunny skies.",
      "Deep learning uses neural networks with many layers to model complex patterns."
    ]
  }'

A successful response should look similar to this:

```json
{
  "results": [
    {"index": 0, "relevance_score": 0.9446689335413089},
    {"index": 2, "relevance_score": 0.31878197585323875},
    {"index": 1, "relevance_score": 0.05886305589012019}
  ]
}
```

Add `top_n` to the JSON request body to limit the number of returned results. See the [rerank API reference](https://docs.zeroentropy.dev/api-reference/models/rerank) for more request options.

## 8. Check service health

The health endpoint returns service status, model configuration, and GPU mode.

In [ ]:
%%bash
set -euo pipefail

curl "http://localhost:${LOCAL_PORT}/health"

Stop the port-forward process when you are done testing.

In [ ]:
if "zerank2_port_forward" in globals() and zerank2_port_forward.poll() is None:
    zerank2_port_forward.terminate()
    zerank2_port_forward.wait(timeout=10)
    print("Stopped port-forward.")
else:
    print("No running port-forward process found.")

## Troubleshooting

- If AKS creation fails, confirm that `AZURE_LOCATION` has quota for both total regional vCPUs and `Standard NCadsH100v5 Family vCPUs`.
- If GPU allocation shows `null`, rerun the GPU verification cell. If it still shows `null`, inspect the NVIDIA device plugin pod in `kube-system`.
- If the Marketplace extension type fails, list the current ZeroEntropy extension registrations with `az k8s-extension extension-types list --cluster-name "$AKS_CLUSTER_NAME" --resource-group "$AZURE_RESOURCE_GROUP" --cluster-type managedClusters`.
- If `kubectl port-forward` fails, rerun the service listing cell and confirm the `zerank-2` service exists.
- If `/invocations` fails, confirm the pod is `1/1 Running`, then inspect logs with the log cell above.

Because GPU-backed AKS clusters are expensive, delete unused test clusters from the Azure portal or with `az aks delete` after you finish.